In [2]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Paramètres
n = 100  # nombre de lignes

# Intervalle de dates
date_debut = datetime(2025, 1, 1)
date_fin = datetime(2026, 3, 31)

# Fonction pour générer une date aléatoire dans l'intervalle
def date_aleatoire():
    delta = date_fin - date_debut
    jours_aleatoires = random.randint(0, delta.days)
    return (date_debut + timedelta(days=jours_aleatoires)).strftime("%Y-%m-%d")

# Listes de données
noms = ["Jean", "Marie", "Paul", "Aline", "David", "Sarah", "Patrick", "Grâce"]
sexes = ["M", "F"]
classes = ["L6", "L5", "L4", "L3", "L2", "L1", "LT"]

data = []

for i in range(1, n + 1):
    ligne = {
        "id": i,
        "nom": random.choice(noms),
        "age": random.randint(18, 25),
        "sexe": random.choice(sexes),
        "classe": random.choice(classes),
        "heures_etude": random.randint(1, 20),
        "absences": random.randint(0, 15),
        "note": random.randint(30, 100),
        "date": date_aleatoire()
    }
    data.append(ligne)

df = pd.DataFrame(data)
df.to_csv("projet_ecole.csv", index=False)

print("Fichier CSV créé")


Fichier CSV créé


In [3]:
import pandas as pd

df = pd.read_csv("projet_ecole.csv")

print(df.head())
print(df.info())
print(df.describe())

# Vérifier valeurs manquantes
print(df.isnull().sum())

   id      nom  age sexe classe  heures_etude  absences  note        date
0   1  Patrick   18    M     L6             3         3    32  2025-08-16
1   2    Sarah   24    F     L4             8         1    60  2026-03-07
2   3    Sarah   24    F     L5            17        13    35  2025-12-19
3   4  Patrick   21    M     L5            11         6    99  2025-04-03
4   5    Aline   23    F     L5             7        11    55  2026-03-15
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            100 non-null    int64 
 1   nom           100 non-null    object
 2   age           100 non-null    int64 
 3   sexe          100 non-null    object
 4   classe        100 non-null    object
 5   heures_etude  100 non-null    int64 
 6   absences      100 non-null    int64 
 7   note          100 non-null    int64 
 8   date          100 non-null  

In [4]:
# Moyenne générale
print("Moyenne générale :", df["note"].mean())

# Moyenne par classe
print(df.groupby("classe")["note"].mean())

# Corrélations
print(df[["note","heures_etude","absences","age"]].corr())

Moyenne générale : 64.87
classe
L1    68.400000
L2    68.875000
L3    65.571429
L4    65.333333
L5    67.466667
L6    58.647059
LT    61.076923
Name: note, dtype: float64
                  note  heures_etude  absences       age
note          1.000000     -0.039864 -0.022102  0.060618
heures_etude -0.039864      1.000000 -0.079610 -0.049297
absences     -0.022102     -0.079610  1.000000 -0.049145
age           0.060618     -0.049297 -0.049145  1.000000


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("projet_ecole.csv")

# Conversion date
df["date"] = pd.to_datetime(df["date"])

# Supprimer doublons
df = df.drop_duplicates()

# Gestion valeurs manquantes
df["heures_etude"] = df["heures_etude"].fillna(df["heures_etude"].median())
df["absences"] = df["absences"].fillna(0)

# Supprimer lignes si note manquante (variable cible)
df = df.dropna(subset=["note"])

In [ ]:
# Détection via IQR - Détection des valeurs aberrantes (Outliers)
Q1 = df["note"].quantile(0.25)
Q3 = df["note"].quantile(0.75)
IQR = Q3 - Q1

borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

df = df[(df["note"] >= borne_inf) & (df["note"] <= borne_sup)]

In [ ]:
df["niveau"] = pd.cut(
    df["note"],
    bins=[0, 60, 75, 90, 100],
    labels=["Faible", "Moyen", "Bon", "Excellent"]
)

In [ ]:
df["engagement"] = df["heures_etude"] / (df["absences"] + 1)

df["risque_echec"] = np.where(
    (df["note"] < 60) & (df["absences"] > df["absences"].mean()),1,0)

df["risque_echec"] = np.where(
    (df["note"] < 60) & (df["absences"] > df["absences"].mean()),1,0)



In [ ]:
df["mois"] = df["date"].dt.month
df["trimestre"] = df["date"].dt.quarter

In [ ]:
print("Corrélations importantes :")
print(df[["note","heures_etude","absences","engagement"]].corr())

print("\nTaux d'échec :")
print(df["risque_echec"].mean())

print("\nMoyenne par niveau :")
print(df.groupby("niveau")["note"].mean())

In [ ]:
from sklearn.model_selection import train_test_split

# Variables numériques
df["sexe_num"] = df["sexe"].map({"F":0, "M":1})
df["classe_num"] = df["classe"].astype("category").cat.codes

features = ["age","sexe_num","classe_num","heures_etude","absences","engagement"]

X = df[features]
y_reg = df["note"]
y_clf = df["risque_echec"]

X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_clf_train, y_clf_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)

In [ ]:
# Modèle 1 : Régression (Prédire la note)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

reg_model = LinearRegression()
reg_model.fit(X_train, y_reg_train)

y_pred_reg = reg_model.predict(X_test)

print("R² :", r2_score(y_reg_test, y_pred_reg))
print("MAE :", mean_absolute_error(y_reg_test, y_pred_reg))

In [ ]:
coefficients = pd.DataFrame({
    "Variable": features,
    "Coefficient": reg_model.coef_
})

print(coefficients.sort_values(by="Coefficient", ascending=False))

In [ ]:
# Modèle 2 : Classification (Prédire le risque)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf_model = LogisticRegression(max_iter=1000)
clf_model.fit(X_train, y_clf_train)

y_pred_clf = clf_model.predict(X_test)

print("Accuracy :", accuracy_score(y_clf_test, y_pred_clf))
print(classification_report(y_clf_test, y_pred_clf))

In [ ]:
importance = pd.DataFrame({
    "Variable": features,
    "Importance": clf_model.coef_[0]
})

print(importance.sort_values(by="Importance", ascending=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.histplot(df["note"], bins=15, kde=True)
plt.title("Distribution des notes")
plt.show()

In [ ]:
sns.scatterplot(x="absences", y="note", data=df)
plt.title("Impact des absences sur la note")
plt.show()

In [ ]:
sns.regplot(x="engagement", y="note", data=df)
plt.title("Relation engagement / note")
plt.show()

In [ ]:
df.groupby("classe")["note"].mean().plot(kind="bar")
plt.title("Moyenne des notes par classe")
plt.show()

In [ ]:
import plotly.express as px

fig1 = px.box(df, x="classe", y="note", title="Distribution des notes par classe")
fig1.show()

fig2 = px.scatter(df, x="heures_etude", y="note", color="classe",
                  title="Heures d'étude vs Note")
fig2.show()